# Day 4 — Frozen penalty × depth experiment

Research question: for the one frozen routing graph, how do the prescribed flow penalty `A` and shallow depth `p ∈ {1,2}` change valid-route and exact-route sampling probability? The optimization protocol is displayed before any saved result is loaded.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import SVG, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'optimization_contract.json').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
RERUN_EXPENSIVE = False  # opt in explicitly; saved results are the default
contract = json.loads((ROOT / 'data' / 'optimization_contract.json').read_text())
contract['optimizer'], contract['experimental_matrix'], contract['start_generation']

## Protocol frozen before results

COBYLA minimizes expected frozen-QUBO energy using exact statevectors. Three seed-10387 starts are generated once per depth and reused for every penalty. Both depths receive at most 240 objective evaluations per start. A cell is selected only by lowest final expected energy, with lower `start_id` as the deterministic tie-breaker—not by `p_feas` or `p_opt`.

In [ ]:
if RERUN_EXPENSIVE:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_day4_core_experiment.py'), '--overwrite'], check=True)
results = json.loads((ROOT / 'results' / 'core_experiment_results.json').read_text())
assert results['optimization_contract']['sha256'] == '12d44320e83ce06ae9172c6d68bd8b0f617bc4fa2b9273418fc0e8bec2ecba3d'
results['experiment_completeness']

## p=2 correctness gate

Before optimization, two deterministic four-parameter vectors at each frozen `A` compared the explicit Qiskit circuit with the independent NumPy tensor-product evolution.

In [ ]:
gate = results['p2_preoptimization_equivalence_gate']
assert gate['passed'] and gate['case_count'] == 8
{key: gate[key] for key in ('minimum_fidelity', 'maximum_amplitude_absolute_error', 'maximum_probability_error', 'maximum_norm_error')}

## Selected cells and primary route metrics

All 24 starts remain in the JSON/CSV artifacts. The compact view below reports only the frozen minimum-energy selection for each of the eight cells.

In [ ]:
table = []
for cell in results['selected_cells']:
    opt, quality = cell['optimization'], cell['quality']
    table.append({'A': cell['identity']['A'], 'p': cell['identity']['p'],
                  'start': opt['selected_start_id'], '<Q_A>': opt['final_expected_qubo_energy'],
                  'p_feas': quality['p_feas'], 'p_opt': quality['p_opt'],
                  'E[C|feasible]': quality['conditional_expected_route_cost']})
table

## Figures 8–11

The landscape is descriptive, not a proof of a global variational optimum. The graph marginals are individual edge-selection probabilities and do not constitute a sampled valid route.

In [ ]:
for name in ('08_p1_variational_landscape.svg', '09_penalty_depth_core_results.svg',
             '10_quantum_route_marginals.svg', '11_quantum_probability_state_space.svg'):
    display(SVG(filename=str(ROOT / 'figures' / name)))

## Observed result under the frozen protocol

Depth `p=2` increased both `p_feas` and `p_opt` at `A=2` and `A=6`, increased only `p_feas` at `A=5`, and decreased both at `A=12`. Penalty behavior was non-monotone overall: `p=1` valid-route probability was non-monotone, while the four prescribed `p=2` valid-route values decreased with `A`; exact-route probability was non-monotone at both depths. Most selected runs exhausted the fixed budget, so all-start sensitivity is part of the result rather than something to tune away.